# 04 Geographical visualisation with Folium

Interactive maps for evaluator_test trips: raw GPS, road network, reference route, classifier-only errors and the classifier + HMM/Viterbi route. Predictions come from `artifacts/evaluator_preds/` (written by `scripts/eval_evaluator_test.py`). The reference route is used only for display and scoring.

In [1]:
from pathlib import Path
import json, sys, warnings
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve().parents[1]          # report/notebooks -> repository root
sys.path.insert(0, str(ROOT / "src"))
FIG, TAB = ROOT / "report" / "figures", ROOT / "report" / "tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
RES = ROOT / "results"
DATA = {"train": ROOT / "student_dataset" / "train", "student_test": ROOT / "student_dataset" / "student_test",
        "evaluator_test": ROOT / "evaluator_test"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "axes.titleweight": "bold"})
PAL = {"train": "#2b6cb0", "student_test": "#dd6b20", "evaluator_test": "#2f855a"}

def save(name):
    plt.savefig(FIG / name, bbox_inches="tight", facecolor="white")
    plt.show()

def read_whitespace(path, dtype=float):
    return pd.read_csv(path, sep=r"\s+", header=None, dtype=dtype).to_numpy()

def load_trip(part, tid):
    d = DATA[part] / tid
    return {"track": read_whitespace(d / f"{tid}.track"), "nodes": read_whitespace(d / f"{tid}.nodes"),
            "arcs": read_whitespace(d / f"{tid}.arcs", dtype=np.int64),
            "route": read_whitespace(d / f"{tid}.route", dtype=np.int64).ravel()}

def trips(part):
    return sorted(p.name for p in DATA[part].iterdir() if p.is_dir() and not p.name.startswith("."))

In [2]:
import folium
from matplotlib.collections import LineCollection
from pathsnap.geo import LocalProjection
PRED = ROOT / "artifacts" / "evaluator_preds"; MAPS = ROOT / "report" / "maps"; MAPS.mkdir(exist_ok=True)
truth = pd.read_parquet(PRED / "truth.parquet"); cand = pd.read_parquet(PRED / "lightgbm_cand.parquet"); hmm = pd.read_parquet(PRED / "lightgbm_point.parquet")
for d in (truth, cand, hmm): d["trip"] = d["trip"].astype(str).str.zfill(8)
top = cand.sort_values(["trip", "pt", "p"], ascending=[True, True, False]).drop_duplicates(["trip", "pt"]).rename(columns={"arc": "clf_arc"})[["trip", "pt", "clf_arc"]]
M = truth.merge(top, on=["trip", "pt"], how="left").merge(hmm.rename(columns={"pred_arc": "hmm_arc"}), on=["trip", "pt"], how="left")
M["clf_ok"] = M.clf_arc == M.truth_arc; M["hmm_ok"] = M.hmm_arc == M.truth_arc
M.groupby("trip")[["clf_ok", "hmm_ok"]].mean().round(4)

,clf_ok,hmm_ok
trip,,
00000015,0.8830,0.9438
00000017,0.9212,0.8559
00000023,0.9476,0.9825
00000030,0.8150,0.8303
00000049,0.9890,0.9860
00000059,0.9253,0.8632
00000067,0.9683,0.9629
00000073,0.9755,0.9764
00000087,0.9585,0.9800


In [3]:
def arc_line(nodes, arcs, a): return [(nodes[arcs[a, 0], 1], nodes[arcs[a, 0], 0]), (nodes[arcs[a, 1], 1], nodes[arcs[a, 1], 0])]
def route_line(nodes, arcs, seq):
    out = []
    for a in seq:
        p = arc_line(nodes, arcs, a)
        if not out or out[-1] != p[0]: out.append(p[0])
        out.append(p[1])
    return out
def dedupe(seq):
    s = np.asarray(seq); return s[np.r_[True, s[1:] != s[:-1]]]

def build_map(tid, name):
    tr = load_trip("evaluator_test", tid); nodes, arcs, track, route = tr["nodes"], tr["arcs"], tr["track"], tr["route"]
    m = M[M.trip == tid].sort_values("pt")
    lat, lon = track[:, 1], track[:, 0]; c = [lat.mean(), lon.mean()]
    fmap = folium.Map(location=c, zoom_start=14, tiles="OpenStreetMap", control_scale=True)
    fmap.fit_bounds([[lat.min(), lon.min()], [lat.max(), lon.max()]])
    pad = 0.004; keep = ((nodes[arcs[:, 0], 1] > lat.min() - pad) & (nodes[arcs[:, 0], 1] < lat.max() + pad) & (nodes[arcs[:, 0], 0] > lon.min() - pad) & (nodes[arcs[:, 0], 0] < lon.max() + pad))
    ids = np.flatnonzero(keep)
    if len(ids) > 6000: ids = ids[:: int(np.ceil(len(ids) / 6000))]
    g_net = folium.FeatureGroup("Road network (grey)", show=True)
    for a in ids: folium.PolyLine(arc_line(nodes, arcs, a), color="#8a94a0", weight=1, opacity=0.7).add_to(g_net)
    g_net.add_to(fmap)
    g_ref = folium.FeatureGroup("Reference route (green)", show=True)
    folium.PolyLine(route_line(nodes, arcs, route), color="#2f855a", weight=7, opacity=0.65, tooltip="Reference route").add_to(g_ref); g_ref.add_to(fmap)
    g_hmm = folium.FeatureGroup("Predicted route: LightGBM + HMM/Viterbi (red)", show=True)
    folium.PolyLine(route_line(nodes, arcs, dedupe(m.hmm_arc.dropna().astype(int))), color="#c53030", weight=3, opacity=0.95, tooltip="LightGBM + HMM route").add_to(g_hmm); g_hmm.add_to(fmap)
    g_gps = folium.FeatureGroup("Raw GPS observations (blue)", show=True)
    for (la, lo, t) in zip(lat, lon, track[:, 2]): folium.CircleMarker((la, lo), radius=2, color="#2b6cb0", fill=True, fill_opacity=0.9, weight=0, tooltip=f"t={t:.0f}s").add_to(g_gps)
    g_gps.add_to(fmap)
    g_wrong = folium.FeatureGroup("Classifier-only errors (orange)", show=True)
    for _, r in m[~m.clf_ok].iterrows():
        folium.CircleMarker((lat[int(r.pt)], lon[int(r.pt)]), radius=4, color="#dd6b20", fill=True, fill_opacity=0.95, weight=1, tooltip=f"point {int(r.pt)}: classifier-only arc is wrong").add_to(g_wrong)
    g_wrong.add_to(fmap)
    folium.Marker((lat[0], lon[0]), tooltip="start", icon=folium.Icon(color="green", icon="play")).add_to(fmap); folium.Marker((lat[-1], lon[-1]), tooltip="end", icon=folium.Icon(color="red", icon="stop")).add_to(fmap)
    legend = f"""<div style="position: fixed; bottom: 24px; left: 24px; z-index: 9999; background: white; padding: 10px 12px; border: 1px solid #888; font-size: 13px; line-height: 1.5">
    <b>Trip {tid} (evaluator_test)</b><br>{len(track)} GPS points<br>
    <span style="color:#2b6cb0">&#9679;</span> raw GPS observation<br><span style="color:#8a94a0">&#9632;</span> road network<br>
    <span style="color:#2f855a">&#9632;</span> reference route<br><span style="color:#c53030">&#9632;</span> predicted route (LightGBM + HMM)<br>
    <span style="color:#dd6b20">&#9679;</span> point wrong without the HMM<br>Top-1: classifier {m.clf_ok.mean():.3f}, with HMM {m.hmm_ok.mean():.3f}</div>"""
    fmap.get_root().html.add_child(folium.Element(legend)); folium.LayerControl(collapsed=False).add_to(fmap)
    fmap.save(MAPS / name); return fmap
gain = (M.groupby("trip").hmm_ok.mean() - M.groupby("trip").clf_ok.mean()); size = M.groupby("trip").size()
big = gain[size > 500].idxmax(); hard = M.groupby("trip").hmm_ok.mean().idxmin()
print("largest HMM gain:", big, round(gain[big], 4), "| hardest trip:", hard)
build_map(big, f"evaluator_trip_{big}.html"); build_map(hard, f"evaluator_trip_{hard}.html")
MAP_TRIPS = {"gain": big, "hard": hard}
MAP_TRIPS
sorted(p.name for p in MAPS.iterdir())

largest HMM gain: 00000015 0.0608 | hardest trip: 00000030


['eda_train_00000046.html',
 'evaluator_trip_00000015.html',
 'evaluator_trip_00000030.html']

## EDA map on a training trip
Training-partition trip 00000046: raw GPS, road network and reference route only (no predictions).

In [4]:
def eda_map(part, tid, name):
    tr = load_trip(part, tid); nodes, arcs, track, route = tr["nodes"], tr["arcs"], tr["track"], tr["route"]; lat, lon = track[:, 1], track[:, 0]
    fmap = folium.Map(location=[lat.mean(), lon.mean()], zoom_start=15, tiles="OpenStreetMap", control_scale=True)
    fmap.fit_bounds([[lat.min(), lon.min()], [lat.max(), lon.max()]])
    pad = 0.003; keep = ((nodes[arcs[:, 0], 1] > lat.min() - pad) & (nodes[arcs[:, 0], 1] < lat.max() + pad) & (nodes[arcs[:, 0], 0] > lon.min() - pad) & (nodes[arcs[:, 0], 0] < lon.max() + pad))
    g1 = folium.FeatureGroup("Road network (grey)"); [folium.PolyLine(arc_line(nodes, arcs, a), color="#8a94a0", weight=1, opacity=0.7).add_to(g1) for a in np.flatnonzero(keep)[:6000]]; g1.add_to(fmap)
    g2 = folium.FeatureGroup("Reference route (green)"); folium.PolyLine(route_line(nodes, arcs, route), color="#2f855a", weight=7, opacity=0.65).add_to(g2); g2.add_to(fmap)
    g3 = folium.FeatureGroup("Raw GPS observations (blue)"); [folium.CircleMarker((la, lo), radius=2, color="#2b6cb0", fill=True, fill_opacity=0.9, weight=0).add_to(g3) for la, lo in zip(lat, lon)]; g3.add_to(fmap)
    legend = f"""<div style="position: fixed; bottom: 24px; left: 24px; z-index: 9999; background: white; padding: 10px 12px; border: 1px solid #888; font-size: 13px; line-height: 1.5">
    <b>Trip {tid} ({part} partition)</b><br>{len(track)} GPS points, {len(route)} reference route arcs<br>
    <span style="color:#2b6cb0">&#9679;</span> raw GPS observation<br><span style="color:#8a94a0">&#9632;</span> road network<br><span style="color:#2f855a">&#9632;</span> reference route</div>"""
    fmap.get_root().html.add_child(folium.Element(legend)); folium.LayerControl(collapsed=False).add_to(fmap); fmap.save(MAPS / name)
eda_map("train", "00000046", "eda_train_00000046.html")
sorted(p.name for p in MAPS.iterdir())

['eda_train_00000046.html',
 'evaluator_trip_00000015.html',
 'evaluator_trip_00000030.html']

## Static comparison (same data, no map tiles)
Reference route, classifier-only route and HMM route for one trip.

In [5]:
tid = MAP_TRIPS["gain"]; tr = load_trip("evaluator_test", tid); proj = LocalProjection(tr["nodes"][:, 0].mean(), tr["nodes"][:, 1].mean())
N = proj.to_xy(tr["nodes"]); P = proj.to_xy(tr["track"][:, :2]); arcs = tr["arcs"]; m = M[M.trip == tid].sort_values("pt")
def seg(seq): s = dedupe(np.asarray(seq).astype(int)); return np.stack([N[arcs[s, 0]], N[arcs[s, 1]]], 1)
fig, ax = plt.subplots(1, 3, figsize=(18, 6.2), sharex=True, sharey=True)
x0, x1, y0, y1 = P[:, 0].min() - 80, P[:, 0].max() + 80, P[:, 1].min() - 80, P[:, 1].max() + 80
A, B = N[arcs[:, 0]], N[arcs[:, 1]]; kp = (A[:, 0] > x0) & (A[:, 0] < x1) & (A[:, 1] > y0) & (A[:, 1] < y1)
for a_, (title, seq, col) in zip(ax, [("Reference route", tr["route"], "#2f855a"), ("Classifier only (LightGBM)", m.clf_arc.dropna(), "#dd6b20"), ("Classifier + HMM/Viterbi", m.hmm_arc.dropna(), "#c53030")]):
    a_.add_collection(LineCollection(np.stack([A[kp], B[kp]], 1), colors="#d0d5db", linewidths=0.5)); a_.add_collection(LineCollection(seg(seq), colors=col, linewidths=2.4))
    a_.scatter(P[:, 0], P[:, 1], s=3, color="#2b6cb0", zorder=3); a_.set_aspect("equal"); a_.set_title(title); a_.grid(False)
ax[0].set_xlim(x0, x1); ax[0].set_ylim(y0, y1); plt.suptitle(f"evaluator_test trip {tid}: GPS (blue) and routes", y=1.0)
plt.tight_layout(); save("map_01_route_comparison_static.png")